# Dasar Machine Learning

## Course 3 Bagian f: Boosting dan Stacking

Latihan ini memakai data Human Activity Recognition. Data berisi 561 fitur hasil pengukuran akselerometer dan giroskop ponsel, dengan enam kelas aktivitas: berjalan, naik tangga, turun tangga, duduk, berdiri, dan berbaring.

### Tujuan

- Menilai pengaruh jumlah pohon pada gradient boosting.
- Mencari hyperparameter terbaik dengan grid search.
- Membandingkan AdaBoost dengan gradient boosting.
- Menggabungkan beberapa model memakai `VotingClassifier`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Soal 1: Memuat data

Sebagian baris dipakai agar pencarian parameter berjalan cepat. Hapus pemanggilan `sample` bila ingin memakai seluruh data.


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/"
    "Human_Activity_Recognition_Using_Smartphones_Data.csv"
)
SAMPLE_SIZE = 2000

activity_data = pd.read_csv(DATA_URL, sep=",").sample(
    n=SAMPLE_SIZE, random_state=42
).reset_index(drop=True)
activity_data.shape


In [ ]:
activity_data.dtypes.value_counts()


In [ ]:
float_columns = activity_data.dtypes == np.float64
print((activity_data.loc[:, float_columns].max() == 1.0).all())
print((activity_data.loc[:, float_columns].min() == -1.0).all())


Seluruh fitur sudah berskala -1 sampai 1. Metode berbasis pohon tidak memerlukan penstandaran tambahan.


## Soal 2: Penyandian target dan pembagian data


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
activity_data["Activity"] = label_encoder.fit_transform(activity_data["Activity"])

feature_columns = [name for name in activity_data.columns if name != "Activity"]
X_train, X_test, y_train, y_test = train_test_split(
    activity_data[feature_columns],
    activity_data["Activity"],
    test_size=0.3,
    random_state=42,
)
X_train.shape, X_test.shape


## Soal 3: Gradient boosting dengan berbagai jumlah pohon

Setiap model dibuat ulang di dalam perulangan karena `warm_start` tidak stabil pada gradient boosting.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

tree_list = [15, 25, 50, 100]
error_rows = []

for n_trees in tree_list:
    boosted = GradientBoostingClassifier(n_estimators=n_trees, random_state=42)
    boosted.fit(X_train, y_train)
    error_rows.append(
        {
            "n_trees": n_trees,
            "error": 1.0 - accuracy_score(y_test, boosted.predict(X_test)),
        }
    )

error_table = pd.DataFrame(error_rows).set_index("n_trees")
error_table


In [ ]:
sns.set_context("talk")
sns.set_style("white")

axis = error_table.plot(marker="o", figsize=(10, 6), linewidth=4)
axis.set(xlabel="Number of Trees", ylabel="Error")
axis.set_xlim(0, max(error_table.index) * 1.1)
plt.tight_layout()
plt.show()


## Soal 4: Grid search gradient boosting

Kombinasi jumlah pohon, learning rate, subsample, dan jumlah fitur diuji dengan validasi silang.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.1, 0.01],
    "subsample": [1.0, 0.5],
    "max_features": [1, 2],
}

boost_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
boost_search.best_estimator_


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = boost_search.predict(X_test)
print(classification_report(y_test, predictions))

sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt="g", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


Model gradient boosting masih kesulitan memisahkan kelas 1 dan 2, yaitu naik dan turun tangga, karena pola sensornya mirip.


## Soal 5: AdaBoost

AdaBoost memakai stump keputusan sebagai estimator dasar. Rentang jumlah estimator diuji bersama dua nilai learning rate.


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_search = GridSearchCV(
    AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), random_state=42),
    param_grid={"n_estimators": [100, 150], "learning_rate": [0.01, 0.1]},
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
ada_search.best_estimator_


In [ ]:
ada_predictions = ada_search.predict(X_test)
print(classification_report(y_test, ada_predictions))

sns.heatmap(confusion_matrix(y_test, ada_predictions), annot=True, fmt="g", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


AdaBoost sangat peka terhadap pencilan, sehingga kesalahan pada kelas 1 dan 2 makin terlihat.


## Soal 6: Regresi logistik dan voting classifier

Regresi logistik dengan penalti L2 dilatih sebagai pembanding, lalu digabung dengan gradient boosting memakai soft voting.


In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    penalty="l2", max_iter=300, solver="saga"
).fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
print(classification_report(y_test, logistic_predictions))


In [ ]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(
    [("logistic", logistic_model), ("boosted", boost_search.best_estimator_)],
    voting="soft",
).fit(X_train, y_train)

voting_predictions = voting_model.predict(X_test)
print(classification_report(y_test, voting_predictions))

sns.heatmap(confusion_matrix(y_test, voting_predictions),
            annot=True, fmt="g", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## Ringkasan

Gradient boosting memberi akurasi tertinggi, sedangkan regresi logistik hampir menyamainya dengan biaya jauh lebih kecil. Voting classifier memperbaiki sedikit hasil kedua model, tetapi keuntungannya tidak selalu sepadan dengan tambahan biaya komputasi.
